# import libraries

In [1]:
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.factory import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy   


import sys
sys.path.append('../')
import helper_functions as hf

# generate recommendations

In [2]:
n = hf.get_iteration_number()

for i in range(3):
    print("Very Important: Please Confirm the Iteration Number is Iteration " + str(n))

Very Important: Please Confirm the Iteration Number is Iteration 4
Very Important: Please Confirm the Iteration Number is Iteration 4
Very Important: Please Confirm the Iteration Number is Iteration 4


In [3]:
df_design, ax_client = hf.run_optimizer(current_iteration=n, n_trials=1)

[INFO 06-18 11:03:52] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 06-18 11:03:52] ax.modelbridge.transforms.standardize_y: Outcome micelle_drug_conc is constant, within tolerance.
/opt/anaconda3/envs/drug_surfactant/lib/python3.11/site-packages/botorch/models/utils/assorted.py:267: InputDataWarning: Data (outcome observations) is not standardized (std = tensor([0.], dtype=torch.float64), mean = tensor([0.], dtype=torch.float64)).Please consider scaling the input to zero mean and unit variance.
  check_standardization(Y=train_Y, raise_on_fail=raise_on_fail)
[INFO 06-18 11:04:41] ax.service.ax_client: Generated new trial 9 with parameters {'s1': 0, 's2': 0, 's3': 100, 's4': 100, 's5': 0, 's6': 100, 's7': 0, 's8': 100, 'surfactant_conc': 1, 'drug_conc': 100} using model SAASBO.
[INFO 06-18 11:04:41] ax.service.ax_client: Saved J

# process results

In [4]:
ax_client = hf.load_design_optimizer(n)
ax_client.experiment.trials

[INFO 06-18 11:04:45] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.


{0: Trial(experiment_name='drug_surfactant', index=0, status=TrialStatus.COMPLETED, arm=Arm(name='0_0', parameters={'s1': 47, 's2': 59, 's3': 49, 's4': 31, 's5': 96, 's6': 8, 's7': 9, 's8': 35, 'surfactant_conc': 85, 'drug_conc': 77})),
 1: Trial(experiment_name='drug_surfactant', index=1, status=TrialStatus.COMPLETED, arm=Arm(name='1_0', parameters={'s1': 95, 's2': 22, 's3': 76, 's4': 63, 's5': 42, 's6': 71, 's7': 55, 's8': 99, 'surfactant_conc': 35, 'drug_conc': 3})),
 2: Trial(experiment_name='drug_surfactant', index=2, status=TrialStatus.COMPLETED, arm=Arm(name='2_0', parameters={'s1': 52, 's2': 98, 's3': 4, 's4': 13, 's5': 22, 's6': 89, 's7': 28, 's8': 56, 'surfactant_conc': 18, 'drug_conc': 33})),
 3: Trial(experiment_name='drug_surfactant', index=3, status=TrialStatus.COMPLETED, arm=Arm(name='3_0', parameters={'s1': 6, 's2': 33, 's3': 71, 's4': 83, 's5': 63, 's6': 25, 's7': 87, 's8': 16, 'surfactant_conc': 68, 'drug_conc': 59})),
 4: Trial(experiment_name='drug_surfactant', inde

In [5]:
df_conc, df_vol = hf.design_to_conc_to_vol (n)

In [6]:
plate_well = input("Enter the plate well starting well (e.g., F1): ")
deepplate_well = input("Enter the deep plate well starting well (e.g., F1): ")


print("Please confirm the following information:")
print("Wellplate will start at: " + plate_well)
print("Deep plate will start at: " + deepplate_well)

print()
print("*******************************************************")
print("Continue if correct, or rerun this cell if incorrect.")
print("*******************************************************")

Please confirm the following information:
Wellplate will start at: D7
Deep plate will start at: G7

*******************************************************
Continue if correct, or rerun this cell if incorrect.
*******************************************************


In [7]:

hf.generate_protocol(df_vol=df_vol, iteration=n, plate_well=plate_well, deepplate_well=deepplate_well)

✅ Successfully wrote to: protocol/otflex_4.py


In [8]:
df_absorbance = hf.process_absorbance(iteration=n, threshold=0.1)
df_absorbance

,trial_index,success
0,0,0


In [9]:
results = hf.build_results(n, df_conc, df_absorbance)
results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_conc,drug_conc,success,micelle_drug_conc,complexity
0,9,0,0,100,100,0,100,0,100,0.5,25.0,0,0.0,4


In [10]:
norm_results = hf.normalize_data(results, 'normalize')

In [11]:
norm_results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_conc,drug_conc,success,micelle_drug_conc,complexity
0,9,0,0,100,100,0,100,0,100,0.01,25.0,0,0.0,0.5


# load the results to the optimizer

In [12]:
ax_client = hf.load_data_to_optimizer(iteration = n, norm_results = norm_results)
ax_client

[INFO 06-18 12:11:04] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 06-18 12:11:04] ax.service.ax_client: Completed trial 9 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (0.01, None), 'complexity': (0.5, None)}.
[INFO 06-18 12:11:05] ax.service.ax_client: Saved JSON-serialized state of optimization to `optimizer/optimizer_4_loaded.json`.


AxClient(experiment=Experiment(drug_surfactant))